# Retina Data Split Notebook

Within this notebook, a core snRNA-seq retina dataset is split between neuronal and non-neuronal cell types and downsampled. The resulting datasets are stored in the same directory as the original under `data_retina_ds_neuron.h5ad` and `data_retina_ds_nonneuron.h5ad`

**Note:** The input data file should be a single-cell RNA-seq dataset in h5ad format, with cell type annotations in the obs dataframe and gene annotations in the var dataframe.

**Note:** If using GitHub for version control and repository sharing, ensure that you add the path to your data folder to the repository's `.gitignore` file, to prevent yourself from exceeding the GitHub's storage limits.

In [1]:
# libraries
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import plotly.io as pio
pio.renderers.default = "notebook"
import nsforest as ns
from nsforest import utils
import celltypist as ct

/opt/miniconda3/envs/nsforest/lib/python3.11/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


In [2]:
# configs
code_folder = "/Users/vbecker/NSForest-ncRNA/beckersv_neural/retina" # path to the NSForest-ncRNA folder
sys.path.insert(0, os.path.abspath(code_folder))

data_folder = "data/" # path to folder containing the input data file (.h5ad format)
file = data_folder + "data_retina_sn.h5ad"

to_downsample = True # True if you want to downsample the dataset to a specific number of cells, 
                     # False otherwise

to_downsample_max = 250000 # Maximum number of cells within each strata's final .h5ad file.

seed = 0 # random seed for reproducibility

In [3]:
# retina specific configs
annotation = 'majorclass'
values = {
    "neuron": ["AC", # amacrine cell
               "BC", # bipolar cell
               "Cone", 
               "HC", # horizontal cell
               "RGC", # retinal ganglion cell
               "Rod"],
    
    "non-neuron": ["Astrocyte",
                   "MG", # Müller glia
                   "Microglia",
                   "RPE"] # retinal pigment epithelium
}

## Data Exploration

### Loading AnnData file

In [4]:
adata_raw = sc.read_h5ad(file, backed = "r")
adata_raw

AnnData object with n_obs × n_vars = 3177310 × 35475 backed at 'data/data_retina_sn.h5ad'
    obs: 'reference_genome', 'gene_annotation_version', 'alignment_software', 'intronic_reads_counted', 'donor_id', 'donor_age', 'self_reported_ethnicity_ontology_term_id', 'donor_cause_of_death', 'donor_living_at_sample_collection', 'sample_id', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_collection_method', 'tissue_source', 'tissue_type', 'suspension_derivation_process', 'suspension_dissociation_reagent', 'suspension_enriched_cell_types', 'suspension_enrichment_factors', 'suspension_uuid', 'suspension_type', 'tissue_handling_interval', 'library_id', 'assay_ontology_term_id', 'sequenced_fragment', 'institute', 'library_id_repository', 'sequencing_platform', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'disease_ontology_term_id', 'reported_diseases', 'sex_ontology_term_id', 'majorclass', 'AC_subclass', 'AC_cluster',

### Looking at sample labels

In [5]:
adata_raw.obs_names # sample names
adata_raw.obs.columns # sample metadata

Index(['reference_genome', 'gene_annotation_version', 'alignment_software',
       'intronic_reads_counted', 'donor_id', 'donor_age',
       'self_reported_ethnicity_ontology_term_id', 'donor_cause_of_death',
       'donor_living_at_sample_collection', 'sample_id',
       'sample_preservation_method', 'tissue_ontology_term_id',
       'development_stage_ontology_term_id', 'sample_collection_method',
       'tissue_source', 'tissue_type', 'suspension_derivation_process',
       'suspension_dissociation_reagent', 'suspension_enriched_cell_types',
       'suspension_enrichment_factors', 'suspension_uuid', 'suspension_type',
       'tissue_handling_interval', 'library_id', 'assay_ontology_term_id',
       'sequenced_fragment', 'institute', 'library_id_repository',
       'sequencing_platform', 'is_primary_data', 'cell_type_ontology_term_id',
       'author_cell_type', 'disease_ontology_term_id', 'reported_diseases',
       'sex_ontology_term_id', 'majorclass', 'AC_subclass', 'AC_cluster',


In [6]:
print(adata_raw.obs[annotation].value_counts()) # gene counts by cell type

majorclass
Rod          1066056
BC            691008
AC            571579
RGC           399605
MG            221612
Cone          127060
HC             80548
Astrocyte      14085
Microglia       4894
RPE              863
Name: count, dtype: int64


### Looking at genes

**Note:** `adata.var_names` must be unique. If there is a problem, usually it can be solved by assigning `adata.var.index = adata.var["ensembl_id"]`. 

In [7]:
print(adata_raw.var_names) # gene names
print(adata_raw.var.columns) # gene metadata

Index(['ENSG00000243485', 'ENSG00000237613', 'ENSG00000186092',
       'ENSG00000239945', 'ENSG00000239906', 'ENSG00000241860',
       'ENSG00000241599', 'ENSG00000286448', 'ENSG00000236601',
       'ENSG00000284733',
       ...
       'ENSG00000275249', 'ENSG00000274792', 'ENSG00000274175',
       'ENSG00000275869', 'ENSG00000273554', 'ENSG00000277836',
       'ENSG00000278633', 'ENSG00000276017', 'ENSG00000278817',
       'ENSG00000277196'],
      dtype='object', length=35475)
Index(['feature_is_filtered', 'feature_name', 'feature_reference',
       'feature_biotype', 'feature_length', 'feature_type'],
      dtype='object')


In [8]:
print(adata_raw.var["feature_type"].value_counts()) # gene counts by feature type

feature_type
protein_coding                        19266
lncRNA                                15491
IG_V_pseudogene                         187
IG_V_gene                               146
TR_V_gene                               106
TR_J_gene                                79
IG_D_gene                                37
TR_V_pseudogene                          33
transcribed_unprocessed_pseudogene       29
transcribed_unitary_pseudogene           19
IG_J_gene                                18
artifact                                 17
IG_C_gene                                14
IG_C_pseudogene                           9
TR_C_gene                                 6
TR_J_pseudogene                           4
TR_D_gene                                 4
processed_pseudogene                      3
IG_J_pseudogene                           3
transcribed_processed_pseudogene          2
TEC                                       1
unprocessed_pseudogene                    1
Name: count, dtype:

## Downsampling by Strata.

### Defining cluster strata header.

In [9]:
cluster_header = "author_cell_type" # column name in adata.obs that contains the cluster labels
                                    # used in NS-Forest

### Checking cell annotation sizes.

**Note:** Some datasets are too large and need to be downsampled to be run through the pipeline. When downsampling, be sure to have all the granular cluster annotations represented. 

In [10]:
pd.DataFrame(adata_raw.obs[cluster_header].value_counts()).reset_index()

,author_cell_type,count
0,Rod,1066056
1,MG,221612
2,MG_OFF,200402
3,MG_ON,151685
4,FMB,144086
...,...,...
118,HAC90,79
119,HAC91,74
120,HAC92,65
121,HAC93,55


### Compute neuron or non-neuron mask.

In [11]:
# compute neuron mask on the adata_raw object
neuron_mask = adata_raw.obs[annotation].isin(values["neuron"])
neuron_idx = np.flatnonzero(neuron_mask)

# temporary backed view
adata_neuron = adata_raw[neuron_mask, :]

In [12]:
# compute non-neuron mask on the adata_raw object
nonneuron_mask = adata_raw.obs[annotation].isin(values["non-neuron"])
nonneuron_idx = np.flatnonzero(nonneuron_mask)

# temporary backed view
adata_nonneuron = adata_raw[nonneuron_mask, :]

### Discovering downsampling threshold per strata.

With `to_downsample_max` value set in config, we'll generate the ideal `to_downsample_n` for each strata to yield a total number of cells close to `to_downsample_max`.

In [13]:
def find_threshold(og_anndata, cluster_header, ideal_total):
    # get counts dataframe
    og_counts = pd.DataFrame(og_anndata.obs[cluster_header].value_counts()).reset_index()
    print(og_counts)
    
    # if there's less cells than the ideal total, just return the number of cells
    if og_counts['count'].sum() <= ideal_total:
        return ideal_total

    # set upper and lower limit, to_downsample_max has ideal
    upper_lim = ideal_total
    lower_lim = int(ideal_total / len(og_counts.index))
    
    # recurse
    return find_threshold_recursive(og_counts, ideal_total, upper_lim, lower_lim)
    
def find_threshold_recursive(count_data, ideal_total, upper_lim, lower_lim):
    # get middle lim
    mid_thresh = (upper_lim + lower_lim) // 2
    
    # get new total
    new_total = count_data['count'].clip(upper=mid_thresh).sum()
    
    # base case check
    if abs(new_total - ideal_total) <= ideal_total * 0.02:
        return mid_thresh
    
    # binary search exhausted :[
    if upper_lim - lower_lim <= 1:
        return mid_thresh

    # sending the recursive case
    if new_total > ideal_total:
        return find_threshold_recursive(
            count_data,
            ideal_total,
            mid_thresh,
            lower_lim
        )
    else:
        return find_threshold_recursive(
            count_data,
            ideal_total,
            upper_lim,
            mid_thresh
        )
    

In [14]:
to_downsample_all = find_threshold(adata_raw, "author_cell_type", 500000)
print(to_downsample_all)

    author_cell_type    count
0                Rod  1066056
1                 MG   221612
2             MG_OFF   200402
3              MG_ON   151685
4                FMB   144086
..               ...      ...
118            HAC90       79
119            HAC91       74
120            HAC92       65
121            HAC93       55
122            HAC95       39

[123 rows x 2 columns]
7454


In [15]:
to_downsample_neuron = find_threshold(adata_neuron, "author_cell_type", 500000)
print(to_downsample_neuron)

    author_cell_type    count
0                Rod  1066056
1             MG_OFF   200402
2              MG_ON   151685
3                FMB   144086
4            ML_Cone   118559
..               ...      ...
114            HAC90       79
115            HAC91       74
116            HAC92       65
117            HAC93       55
118            HAC95       39

[119 rows x 2 columns]
7589


In [16]:
to_downsample_nonneuron = find_threshold(adata_nonneuron, "author_cell_type", 500000)
print(to_downsample_nonneuron)

  author_cell_type   count
0               MG  221612
1        Astrocyte   14085
2        Microglia    4894
3              RPE     863
500000


### (Optional) Downsampling.

**Note:** Clusters with less cells than specified in `to_downsample_n` will have all cells sampled.

In [17]:
adata_raw._raw = None # fix an issue with original data that prevented writing

In [18]:
# downsample for all cell types
idx = ct.samples.downsample_adata(
    adata_raw,
    mode="each",
    by=cluster_header,
    n_cells=to_downsample_all,
    random_state=seed,
    return_index=True,
)

# index into the original backed object
adata_raw[idx, :].to_memory().write_h5ad(filename = data_folder + "data_retina_ds_all.h5ad")

In [19]:
# downsample within the neuron view
idx = ct.samples.downsample_adata(
    adata_neuron,
    mode="each",
    by=cluster_header,
    n_cells=to_downsample_neuron,
    random_state=seed,
    return_index=True,
)

# convert indices relative to the neuron view -> original adata
orig_idx = neuron_idx[idx]

# index into the original backed object
adata_raw[orig_idx, :].to_memory().write_h5ad(filename = data_folder + "data_retina_ds_neuron.h5ad")

In [20]:
# downsample within the non-neuron view
idx = ct.samples.downsample_adata(
    adata_nonneuron,
    mode="each",
    by=cluster_header,
    n_cells=to_downsample_nonneuron,
    random_state=seed,
    return_index=True,
)

# convert indices relative to the neuron view -> original adata
orig_idx = neuron_idx[idx]

# index into the original backed object
adata_raw[orig_idx, :].to_memory().write_h5ad(filename = data_folder + "data_retina_ds_nonneuron.h5ad")